<a href="https://colab.research.google.com/github/GandharvaThite/ADM-Project/blob/main/Approach_1_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import zscore
from sklearn.model_selection import KFold
from xgboost import XGBRFClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold,StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/cicids2017_curated_38cols_Binary_Latest.csv")

In [ ]:
df

,Avg Bwd Segment Size,Bwd Packet Length Mean,Average Packet Size,Total Length of Bwd Packets,Subflow Bwd Bytes,Packet Length Variance,Packet Length Std,Bwd Packet Length Max,Packet Length Mean,Max Packet Length,...,Bwd IAT Max,Fwd Packets/s,Flow Packets/s,Fwd Header Length,Fwd Header Length.1,Flow IAT Mean,Active Mean,Active Min,Active Max,Label
0,6.000000,6.000000,9.000000,6,6,0.000000,0.000000,6,6.000000,6,...,0,26.104208,52.208416,20,20,38308.000000,0.0,0,0,0
1,65.200000,65.200000,31.125000,326,326,3195.595588,56.529599,163,29.294118,163,...,237,22964.509390,33402.922760,368,368,31.933333,0.0,0,0,0
2,525.000000,525.000000,393.750000,3150,3150,451250.132400,671.751541,1575,370.588235,1575,...,810,9132.420091,14611.872150,336,336,73.000000,0.0,0,0,0
3,555.000000,555.000000,348.689655,6660,6660,496537.374700,704.654082,3069,337.066667,3069,...,13961,1117.979745,1907.141918,560,560,543.071429,0.0,0,0,0
4,525.333333,525.333333,420.133333,3152,3152,496440.116700,704.585067,1576,393.875000,1576,...,794,8241.758242,13736.263740,304,304,78.000000,0.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2233159,0.000000,0.000000,0.000000,0,0,0.000000,0.000000,0,0.000000,0,...,0,500000.000000,500000.000000,64,64,4.000000,0.0,0,0,0
2233160,0.000000,0.000000,0.000000,0,0,0.000000,0.000000,0,0.000000,0,...,0,45454.545450,90909.090910,32,32,22.000000,0.0,0,0,0
2233161,0.000000,0.000000,0.000000,0,0,0.000000,0.000000,0,0.000000,0,...,0,22222.222220,44444.444440,32,32,45.000000,0.0,0,0,0
2233162,0.000000,0.000000,9.000000,0,0,0.000000,0.000000,0,6.000000,6,...,0,41666.666670,41666.666670,40,40,48.000000,0.0,0,0,0


In [ ]:
X = df.drop(columns=['Label'])
Y = df['Label']
scaler = StandardScaler()

In [ ]:
def get_model_score(Model,X_train_trf, X_test_trf,Y_train,Y_test):

  Model.fit(X_train_trf,Y_train)
  Y_pred = Model.predict(X_test_trf)
  accuracy = accuracy_score(Y_test, Y_pred)
  precision = precision_score(Y_test, Y_pred,average = None)
  recall = recall_score(Y_test, Y_pred,average = None)
  f1 = f1_score(Y_test, Y_pred,average = None)

  return accuracy, precision, recall, f1

In [ ]:
def Scores_Stratified(X,Y,Model):
  folds = StratifiedShuffleSplit(n_splits=5, test_size=0.5, random_state=0)
  acc_arr = []
  pre_arr = []
  re_arr = []
  f_arr = []
  combined = []

  for train_index, test_index in folds.split(X,Y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    Y_train, Y_test = Y.iloc[train_index], Y.iloc[test_index]
    X_train_trf = scaler.fit_transform(X_train)
    X_test_trf = scaler.transform(X_test)
    acc,pre,re,f = get_model_score(Model,X_train_trf,X_test_trf,Y_train,Y_test)
    acc_arr.append(acc)
    pre_arr.append(pre)
    re_arr.append(re)
    f_arr.append(f)
  combined.append(acc_arr)
  combined.append(pre_arr)
  combined.append(re_arr)
  combined.append(f_arr)

  return combined

In [ ]:
def average_scores(combined_m):
  avg_acc = 0

  avg_pre_zero = 0
  avg_recall_zero = 0
  avg_f1_zero = 0

  avg_pre_one = 0
  avg_recall_one = 0
  avg_f1_one = 0

  for i in range(len(combined_m[0])):
    avg_acc+=combined_m[0][i]
    avg_pre_zero = combined_m[1][0][0] + combined_m[1][1][0] + combined_m[1][2][0] + combined_m[1][3][0] + combined_m[1][4][0]
    avg_pre_one = combined_m[1][0][1] + combined_m[1][1][1] + combined_m[1][2][1] + combined_m[1][3][1] + combined_m[1][4][1]

    avg_recall_zero = combined_m[2][0][0] + combined_m[2][1][0] + combined_m[2][2][0] + combined_m[2][3][0] + combined_m[2][4][0]
    avg_recall_one = combined_m[2][0][1] + combined_m[2][1][1] + combined_m[2][2][1] + combined_m[2][3][1] + combined_m[2][4][1]

    avg_f1_zero = combined_m[3][0][0] + combined_m[3][1][0] + combined_m[3][2][0] + combined_m[3][3][0] + combined_m[3][4][0]
    avg_f1_one = combined_m[3][0][1] + combined_m[3][1][1] + combined_m[3][2][1] + combined_m[3][3][1] + combined_m[3][4][1]

  report = [avg_acc/5,avg_pre_zero/5,avg_pre_one/5,avg_recall_zero/5,avg_recall_one/5,avg_f1_zero/5,avg_f1_one/5]
  return report

In [ ]:
def display_report(report):
  print("Accuracy: ",report[0])
  print("----------------------------------------------------------------------------------------")
  print("Precision")
  print("Class 0: ",report[1],"              Class 1: ",report[2])
  print("----------------------------------------------------------------------------------------")
  print("Recall")
  print("Class 0: ",report[3],"              Class 1: ",report[4])
  print("----------------------------------------------------------------------------------------")
  print("F1-Score")
  print("Class 0: ",report[5],"              Class 1: ",report[6])
  print("----------------------------------------------------------------------------------------")


In [ ]:
# Model-1
combined_lr = Scores_Stratified(X,Y,LogisticRegression())

In [ ]:
# Model-2
combined_nb = Scores_Stratified(X,Y,GaussianNB())

In [ ]:
# Model-3
combined_dt = Scores_Stratified(X,Y,DecisionTreeClassifier())

In [ ]:
# Model-4
combined_rf = Scores_Stratified(X,Y,RandomForestClassifier(n_estimators=50,max_depth=10))

In [ ]:
report_lr = average_scores(combined_lr)
report_nb = average_scores(combined_nb)
report_dt = average_scores(combined_dt)
report_rf = average_scores(combined_rf)

In [ ]:
# Display performance metrics of model - 1
display_report(report_lr)

In [ ]:
# Display performance metrics of model - 2
display_report(report_nb)

In [ ]:
# Display performance metrics of model - 3
display_report(report_dt)

In [ ]:
# Display performance metrics of model - 4
display_report(report_rf)